In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
        """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(f"{i} : {chunk.page_content}")

0 : 환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.


In [9]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

API_KEY = os.getenv("NVIDIA")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")

embedding = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print("documents : ", len(documents))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

print("chunk : ", len(chunks))

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

print("FIASS 생성완료")

query = "펀드가 무엇인가요?"

result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print("결과 : ", i)
    print("페이지 : ", doc.metadata.get("page"))
    print()
    print(doc.page_content)
    print()

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


documents :  191
chunk :  332
FIASS 생성완료
결과 :  0
페이지 :  31

것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 자금은 예금자보호대상이 아닙니다. 
그러나 펀드의 경우 투자자들의 자금으로 취득한 펀드재산은 자산운용회사의 고유재산과 분리되어 
신탁업자가 별도로 관리하기 때문에 자산운용회사가 파산하더라도 펀드내의 집합투자재산은 안전하다고 
할 수 있습니다.

결과 :  1
페이지 :  28

ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기

In [14]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")

PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(api_key=API_KEY, model=MODEL)

embeddings = NVIDIAEmbeddings(api_key=API_KEY, model=EMBEDDING_MODEL)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]{question}",
        ),
    ]
)

parser = StrOutputParser()

# question = "펀드란 무엇인가요?"
question = "대한민국의 수도는 어디인가요?"

retriever_docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in retriever_docs])

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result : ", result)

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


result :  제공된 문서에서 답을 찾을 수 없습니다.


In [17]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

questions = [
    "펀드란 무엇인가요?",
    "대한민국의 수도는 어디인가요?",
]

for question in questions:
    result = vectorstore.similarity_search_with_score(question, k=3)

    print("질문 : ", question)

    for i, (doc, score) in enumerate(result):
        print("검색 결과")
        print("distance : ", score)
        print("page : ", doc.metadata.get("page"))
        print("내용 : ", doc.page_content[:500])

python-dotenv could not parse statement starting at line 30


/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


질문 :  펀드란 무엇인가요?
검색 결과
distance :  1.2219472
page :  31
내용 :  것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니다.
Q.
A.
펀드도 예금자보호대상이 되나요?
펀드는 예금이 아니라 실적배당상품이므로 펀드에 가입된 투자자들의 
검색 결과
distance :  1.2478447
page :  28
내용 :  ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기대할 수 있다는 장점이 있습니다.
나. 본인의 투자성향, 목표 등의 확인 
•나의 투자성향과 목표는? : 펀드에 가입할 때에는 높은 수익만을 기대하고 무작정 가입하는 것보다는 본인의
투자성향, 

In [18]:
import os
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

load_dotenv()

API_KEY = os.getenv("NVIDIA")
MODEL = os.getenv("NVIDIA_MODEL")
EMBEDDING_MODEL = os.getenv("NVIDIA_EMBEDDING")
PDF_PATH = "./rag_data/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatNVIDIA(model=MODEL, api_key=API_KEY)

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, api_key=API_KEY)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드의 종류와 특징은 무엇인가요?"

similarity_retriever = vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 3}
)

similarity_docs = similarity_retriever.invoke(question)

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10,
        "lambda_mult": 0.5,
    },
)

mmr_docs = mmr_retriever.invoke(question)

print("\n")
print("=" * 80)
print("Similarity Search 결과")
print("=" * 80)

for i, doc in enumerate(similarity_docs):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print(doc.page_content[:500])


# ============================================================
# 11. MMR 결과
# ============================================================

print("\n")
print("=" * 80)
print("MMR Search 결과")
print("=" * 80)

for i, doc in enumerate(mmr_docs):
    print(f"\n----- 결과 {i + 1} -----")

    print("Page:", doc.metadata.get("page"))

    print(doc.page_content[:500])

python-dotenv could not parse statement starting at line 30
/Users/parkchanryong/Desktop/SK네트웍스/SKN_35/llm_workspace/.venv/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(




Similarity Search 결과

----- 결과 1 -----
Page: 28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집28
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
▒ 펀드 가입전 정보수집
가. 펀드 상품의 특성 이해 
•원금손실 발생 가능 : 펀드는 원금이 보장되지 않는 실적배당형 상품입니다. 운용결과에 따라 수익이 제한없이
늘어날 수 있지만, 원금손실이 발생할 수도 있다는 점에 유의하여야 합니다.
•펀드상품의 장점 : 일정 수준까지 원금 및 이자를 보장해주는 은행권의 예·적금에 비해 안정성 측면에서는
불리 하지만, 운용성과에 따라 높은 수익을 기대할 수 있다는 장점이 있습니다.
나. 본인의 투자성향, 목표 등의 확인 
•나의 투자성향과 목표는? : 펀드에 가입할 때에는 높은 수익만을 기대하고 무작정 가입하는 것보다는 본인의
투자성향, 투자 목표 등을 검토해 본 후 가입하는 것이 좋습니다. 
•투자성향이란? :  수익 및 투자위험에 대한 본인의 기대 수준을 말합니다. 높은 수익을 위해서 손실이 발생해도
감내할 수 있는지, 아니

----- 결과 2 -----
Page: 30
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집30
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
다. 적합한 펀드 선정
•투자권유 청취 및 상품선정 : 투자자 유형에 대한 분류결과에 기초하여 본인에게 적정한 투자권유를 받고 적합 
한 상품을 선정합니다. 인터넷을 통해 가입할 경우 전자서명의 방법으로 확인서에 서명하는 방식으로 가입이 진행 
됩니다. 투자권유를 원치 않거나 본인 투자유형 등급보다 높은 등급의 펀드투자를 원할 경우 투자자 확인서에 
서명 후 투자 가능합니다.
라. 펀드에 대한 설명 청취
•펀드에 대한 설명 청취 : 투자권유 펀드의 투자대상자산 등 운용전략, 원본손실위험 등 투자위험, 보수·수수료
및 펀드운용비용, 환매방법 등에 대한 설명을 판매회사의 직원에게 듣습니다. 
마. 투자자 의사확인 후 가입절차
•판매회